In [41]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/one-million-clicks-later/sample_submission.csv
/kaggle/input/one-million-clicks-later/train.csv
/kaggle/input/one-million-clicks-later/test.csv


In [42]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score
import lightgbm as lgb


In [43]:
train = pd.read_csv("/kaggle/input/one-million-clicks-later/train.csv")
test = pd.read_csv("/kaggle/input/one-million-clicks-later/test.csv")
sample = pd.read_csv("/kaggle/input/one-million-clicks-later/sample_submission.csv")


print(train.shape, test.shape)
train.head()


(610310, 15) (152578, 14)


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,user_id,video_id,video_duration,watch_time,liked,commented,subscribed_after,category,device,watch_time_of_day,recommended,clicked,timestamp,watch_percent,id
0,59445,40936,1134.886052,1287.412446,1,0,0,Sports,Tablet,Afternoon,1,0.0,2025-09-07 02:10:34,1.000000,283860
1,55829,17468,1335.223001,1224.760878,0,0,0,Gaming,Mobile,Night,1,0.0,2025-09-22 04:16:24,NaN,632997
2,68379,41436,2880.210321,1506.440934,1,0,0,Comedy,Desktop,Morning,0,0.0,2025-09-14 13:48:16,0.588424,94152
3,70789,17131,2975.577309,2327.012776,0,0,0,Comedy,TV,Evening,0,0.0,2024-01-21 03:33:17,NaN,483728
4,15748,1956,1022.594859,1041.854002,0,0,0,Gaming,Tablet,Evening,0,0.0,2023-02-03 19:40:06,0.978261,189031


In [44]:
train = train.dropna(subset=['clicked'])
train['clicked'] = train['clicked'].astype(int)
test = test.fillna(0)


In [45]:

for col in ['liked', 'commented', 'subscribed_after']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
        
for col in ['liked', 'commented', 'subscribed_after']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().map({
            'yes': 1, 'true': 1, '1': 1,
            'no': 0, 'false': 0, '0': 0
        }).fillna(0).astype(int)



In [52]:
for df in [train, test]:
    
    for col in ['liked', 'commented', 'subscribed_after']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

  
    df['watch_ratio'] = df['watch_time'] / (df['video_duration'] + 1e-5)
    df['is_short_video'] = (df['video_duration'] < 60).astype(int)
    df['engagement_score'] = df['liked'] + df['commented'] + df['subscribed_after']

    if 'clicked' in df.columns:
        df['recommended_interaction'] = df['recommended'] * df['clicked']
    else:
        df['recommended_interaction'] = df['recommended']

    df['category_device_combo'] = df['category'].astype(str) + "_" + df['device'].astype(str)



In [ ]:
def feature_engineering(df):
    num_cols = df.select_dtypes(include=['number']).columns
    df[num_cols] = df[num_cols].fillna(0)
    df.columns = df.columns.str.lower().str.strip()
    if 'watch_time' in df.columns and 'video_duration' in df.columns:
        df['watch_ratio'] = df['watch_time'] / (df['video_duration'] + 1e-5)
        df['is_short_video'] = (df['video_duration'] < 60).astype(int)
    else:
        df['watch_ratio'] = 0
        df['is_short_video'] = 0
    for col in ['liked', 'commented', 'subscribed_after']:
        if col not in df.columns:
            df[col] = 0
    df['engagement_score'] = df['liked'] + df['commented'] + df['subscribed_after']
    if 'recommended' in df.columns:
        if 'clicked' in df.columns:
            df['recommended_interaction'] = df['recommended'] * df['clicked']
        else:
            df['recommended_interaction'] = df['recommended']
    else:
        df['recommended_interaction'] = 0
    if 'category' in df.columns and 'device' in df.columns:
        df['category_device_combo'] = df['category'].astype(str) + "_" + df['device'].astype(str)
    else:
        df['category_device_combo'] = "unknown"
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        df[col] = df[col].astype(str).fillna('unknown')
        df[col] = df[col].astype('category').cat.codes
    if 'watch_ratio' in df.columns and 'engagement_score' in df.columns:
        df['ratio_x_engagement'] = df['watch_ratio'] * df['engagement_score']
    if 'video_duration' in df.columns:
        df['log_duration'] = np.log1p(df['video_duration'])
    for col in ['watch_ratio', 'watch_time', 'video_duration']:
        if col in df.columns:
            q1, q3 = df[col].quantile([0.05, 0.95])
            df[col] = df[col].clip(q1, q3)
    for col in ['id', 'user_id', 'video_id']:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)
    return df

train = feature_engineering(train)
test = feature_engineering(test)

print("Feature engineering done")
print("Train:", train.shape, "| Test:", test.shape)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

X = train.drop(['clicked', 'timestamp'], axis=1, errors='ignore')
y = train['clicked']

X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
test = test.replace([np.inf, -np.inf], np.nan).fillna(0)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
test_scaled = scaler.transform(test.drop(['timestamp'], axis=1, errors='ignore'))


In [ ]:
import lightgbm as lgb
from sklearn.metrics import f1_score

model = lgb.LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.02,
    num_leaves=60,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='binary_logloss',
    callbacks=[lgb.early_stopping(100)]
)

y_pred_proba = model.predict_proba(X_val)[:, 1]

best_f1 = 0
best_thresh = 0
for t in np.linspace(0.1, 0.9, 100):
    f1 = f1_score(y_val, (y_pred_proba > t).astype(int))
    if f1 > best_f1:
        best_f1, best_thresh = f1, t

print("Best Threshold:", best_thresh)
print("Best Validation F1:", best_f1)


In [56]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import lightgbm as lgb
import pandas as pd
import numpy as np

X = train.drop(['clicked', 'timestamp', 'id'], axis=1, errors='ignore')
y = train['clicked']

X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
test = test.replace([np.inf, -np.inf], np.nan).fillna(0)

cat_cols = X.select_dtypes(include=['object']).columns
for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([X[col], test[col]]).astype(str))
    X[col] = le.transform(X[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

num_cols = X.select_dtypes(include=['int64', 'float64']).columns
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])
test[num_cols] = scaler.transform(test[num_cols])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

params = {
    'objective': 'binary',
    'metric': 'binary_error',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1
}

callbacks = [lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=50)]

model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_train, lgb_val],
    num_boost_round=300,
    callbacks=callbacks
)

preds = model.predict(test.drop(['timestamp', 'id'], axis=1, errors='ignore'))
submission = pd.DataFrame({
    'id': test['id'],
    'clicked': (preds > 0.5).astype(int)
})
submission.to_csv("submission.csv", index=False)
submission.head()


Training until validation scores don't improve for 50 rounds
[50]	training's binary_error: 0.0882461	valid_1's binary_error: 0.0885206
Early stopping, best iteration is:
[11]	training's binary_error: 0.0882461	valid_1's binary_error: 0.0885206


,id,clicked
0,53363,0
1,293669,0
2,52195,1
3,260007,1
4,602213,1
